# Reload a checkpoint and explore sampling

Use the run created by [train-gpt.ipynb](train-gpt.ipynb), or point `RUN_PATH` at a run from [finetune-gpt.ipynb](finetune-gpt.ipynb). No training and no model download are needed for the default scratch checkpoint.

**You will:** restore the exact tokenizer and architecture, check a prompt, and compare sampling settings. jadeGPT builds on Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT).


## Environment and reproducibility

Use the environment described in the [README](README.md); Python 3.13 is the recommended starting point. From the repository root, run:

```sh
python -m pip install -e ".[notebooks]"
python -m jupyter lab
```

Choose the Python kernel from that environment and run cells **in order**. Installation is a one-time terminal step, not part of the lesson. Paths below work on Windows, macOS, and Linux. `auto` selects an available accelerator or CPU, then a supported precision. A seed makes comparisons more useful, but exact results can differ across hardware and library versions.


In [ ]:
from pathlib import Path
import json
import os
import sys

import torch
import jadegpt

ROOT = Path.cwd().resolve()
if not (ROOT / "jadegpt.py").is_file():
    raise RuntimeError("Start JupyterLab from the jadeGPT repository root.")

SEED = 1337
DEVICE = jadegpt.resolve_device("auto")
DTYPE = jadegpt.resolve_dtype("auto", DEVICE)
# A few CPU threads often suit a tiny model better than every available core.
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")


## 1. Restore a matched model and tokenizer

Token IDs are arbitrary labels. Two character tokenizers can have the same size and still assign different IDs, so a vocabulary-size check alone cannot establish compatibility. Use the metadata from the same run. Paths in `run.json` are relative to this repository and work across operating systems.


In [ ]:
RUN_PATH = ROOT / "artifacts" / "notebooks" / "scratch" / "run.json"
# To sample your adapted model, replace "scratch" above with "finetune".
if not RUN_PATH.is_file():
    raise FileNotFoundError("Run train-gpt.ipynb first, or select an existing run.json.")

run = json.loads(RUN_PATH.read_text(encoding="utf-8"))
CHECKPOINT_PATH = ROOT / run["checkpoint_path"]
META_PATH = ROOT / run["meta_path"]
metadata = jadegpt.load_metadata(META_PATH)
model = jadegpt.resume_gpt(CHECKPOINT_PATH, random_seed=SEED, device=DEVICE)
USE_GPT2_ENCODING = metadata["encoding"] == "gpt2"
if USE_GPT2_ENCODING:
    assert model.config.vocab_size >= metadata["actual_vocab_size"]
else:
    assert model.config.vocab_size == metadata["vocab_size"], "Model/tokenizer vocabulary mismatch."
print(f"Checkpoint: {CHECKPOINT_PATH.name}")
print(f"Tokenizer: {metadata['encoding']} | context window: {model.config.block_size}")


## 2. Choose a prompt and check its alphabet

The default corpus uses lowercase letters. A character model cannot encode a new character simply because it looks familiar to us. To add new characters, train a model with a new vocabulary, or deliberately implement vocabulary expansion; silently replacing the tokenizer corrupts the meaning of existing weights.


In [ ]:
PROMPT = "the "
if not USE_GPT2_ENCODING:
    unknown = sorted(set(PROMPT) - set(metadata["stoi"]))
    if unknown:
        raise ValueError(f"Prompt contains characters outside this tokenizer: {unknown!r}")
print(f"Prompt: {PROMPT!r}")


## 3. Compare temperature with a controlled seed

Both runs begin with the same random seed and prompt, while temperature changes. A lower temperature usually yields more predictable continuations. Very low temperature may amplify repetition; high temperature can destroy spelling in a small character model. `top_k=20` keeps at most twenty candidate tokens at each step.

The output includes the prompt. The model uses its most recent context window when a sequence becomes longer than `block_size`; a long output does not imply that it remembers the entire output at once.


In [ ]:
samples = {}
for temperature in (0.5, 1.0):
    torch.manual_seed(SEED)
    samples[temperature] = jadegpt.generate_text(
        model=model, start=PROMPT, use_gpt2_encoding=USE_GPT2_ENCODING,
        meta_dir=META_PATH.parent, meta_file_name=META_PATH.name,
        num_samples=1, max_new_tokens=120, temperature=temperature, top_k=20,
        device=DEVICE, dtype=DTYPE,
    )
    print(f"\n--- temperature {temperature} ---\n{samples[temperature]}")


## Experiments and interpretation

- Keep temperature fixed and change only the seed. Which patterns are stable, and which details vary?
- Set `top_k=1`. Why does changing temperature no longer change the selected highest-probability token?
- Compare an early and a later checkpoint with the same prompt and sampling settings. Do both loss and readable structure improve?
- Try a prompt longer than the context window. Explain which part of it can influence the first generated token.

This tiny model is a next-token predictor. Its output can be repetitive, incorrect, or copied from the training text. For the scratch lesson, a successful run means that the paired artifacts reload and produce valid characters; polished prose is a separate modeling objective.
